In [4]:
import requests

In [17]:
api_json={
  "2": {
    "inputs": {
      "width": 832,
      "height": 1216,
      "batch_size": 1
    },
    "class_type": "EmptyLatentImage",
    "_meta": {
      "title": "空Latent图像"
    }
  },
  "4": {
    "inputs": {
      "text": "ocean, tree, sunset, car, (masterpiece, best quality,newest,absurdres,highres)",
      "clip": [
        "7",
        1
      ]
    },
    "class_type": "CLIPTextEncode",
    "_meta": {
      "title": "CLIP文本编码"
    }
  },
  "5": {
    "inputs": {
      "text": "worst quality, old, early, low quality, lowres, signature, username, logo, bad hands, mutated hands",
      "clip": [
        "7",
        1
      ]
    },
    "class_type": "CLIPTextEncode",
    "_meta": {
      "title": "CLIP文本编码"
    }
  },
  "6": {
    "inputs": {
      "seed": 67,
      "steps": 28,
      "cfg": 5,
      "sampler_name": "euler",
      "scheduler": "simple",
      "denoise": 1,
      "model": [
        "7",
        0
      ],
      "positive": [
        "4",
        0
      ],
      "negative": [
        "5",
        0
      ],
      "latent_image": [
        "2",
        0
      ]
    },
    "class_type": "KSampler",
    "_meta": {
      "title": "K采样器"
    }
  },
  "7": {
    "inputs": {
      "ckpt_name": "noobaiXLNAIXL_vPred10Version.safetensors"
    },
    "class_type": "CheckpointLoaderSimple",
    "_meta": {
      "title": "Checkpoint加载器（简易）"
    }
  },
  "8": {
    "inputs": {
      "samples": [
        "6",
        0
      ],
      "vae": [
        "7",
        2
      ]
    },
    "class_type": "VAEDecode",
    "_meta": {
      "title": "VAE解码"
    }
  },
  "9": {
    "inputs": {
      "images": [
        "8",
        0
      ]
    },
    "class_type": "PreviewImage",
    "_meta": {
      "title": "预览图像"
    }
  },
  "14": {
    "inputs": {
      "filename_prefix": "api_result",
      "images": [
        "8",
        0
      ]
    },
    "class_type": "SaveImage",
    "_meta": {
      "title": "保存图像"
    }
  }
}

In [18]:
response = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": api_json})

In [19]:
response.raise_for_status()

In [20]:
print(response.text)
data = response.json()
assert not data.get('node_errors')
prompt_id = data['prompt_id']
print(prompt_id)

{"prompt_id": "6d5f1d4e-02e2-4b8b-afd3-eb0ee0e44050", "number": 4, "node_errors": {}}
6d5f1d4e-02e2-4b8b-afd3-eb0ee0e44050


In [25]:
response = requests.get(f"http://127.0.0.1:8188/history/{prompt_id}")
response.raise_for_status()

response.text

'{"6d5f1d4e-02e2-4b8b-afd3-eb0ee0e44050": {"prompt": [4, "6d5f1d4e-02e2-4b8b-afd3-eb0ee0e44050", {"2": {"inputs": {"width": 832, "height": 1216, "batch_size": 1}, "class_type": "EmptyLatentImage", "_meta": {"title": "\\u7a7aLatent\\u56fe\\u50cf"}}, "4": {"inputs": {"text": "ocean, tree, sunset, car, (masterpiece, best quality,newest,absurdres,highres)", "clip": ["7", 1]}, "class_type": "CLIPTextEncode", "_meta": {"title": "CLIP\\u6587\\u672c\\u7f16\\u7801"}}, "5": {"inputs": {"text": "worst quality, old, early, low quality, lowres, signature, username, logo, bad hands, mutated hands", "clip": ["7", 1]}, "class_type": "CLIPTextEncode", "_meta": {"title": "CLIP\\u6587\\u672c\\u7f16\\u7801"}}, "6": {"inputs": {"seed": 67, "steps": 28, "cfg": 5.0, "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0, "model": ["7", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["2", 0]}, "class_type": "KSampler", "_meta": {"title": "K\\u91c7\\u6837\\u5668"}}, "7": {"inputs": {"c

In [31]:

import re

def find_output_images(data, pattern):
    """
    递归遍历 JSON，寻找所有符合 type='output' 且 filename 匹配 pattern 的字典对象。
    """
    results = []
    
    if isinstance(data, dict):
        # 健壮性检查：判断当前字典是否同时包含 filename 和 type
        if "filename" in data and "type" in data:
            if data["type"] == "output" and re.match(pattern, str(data["filename"])):
                # 找到目标，返回包含完整信息的字典
                results.append({
                    "filename": data["filename"],
                    "subfolder": data.get("subfolder", ""),
                    "type": data["type"]
                })
        
        # 继续递归遍历字典的所有值
        for value in data.values():
            results.extend(find_output_images(value, pattern))
            
    elif isinstance(data, list):
        # 遍历列表项
        for item in data:
            results.extend(find_output_images(item, pattern))
            
    return results


my_results = find_output_images(response.json(), r"^api_result_.*\.png$")

print("my_results:", my_results)

assert len(my_results) == 1

my_results: [{'filename': 'api_result_00001_.png', 'subfolder': '', 'type': 'output'}]


In [34]:
def download_image(img_info, base_url="http://localhost:8188"):
    params = {
        "filename": img_info["filename"],
        "subfolder": img_info["subfolder"],
        "type": img_info["type"]
    }
    
    response = requests.get(f"{base_url}/view", params=params)
    response.raise_for_status()
    
    with open('api.note.tmp.png', "wb") as f:
        f.write(response.content)

download_image(my_results[0])